In [7]:
# Imports and Constants
import pandas as pd
import numpy as np
import sqlite3
import joblib
import torch
from torch import Tensor

from climb_conversion import ClimbsFeatureScaler, NUM_ROLES, NUM_HOLD_FEATURES
from unet_diffusion import ClimbDDPM, Noiser

DB_PATH = "data/storage.db"
SCALER_WEIGHTS_PATH = 'data/weights/scaler-weights.joblib'
DDPM_WEIGHTS_PATH = 'data/weights/ddpm-weights.pth'
LAYOUT_ID = 'layout-0aa86d03949f'


GRADE_TO_DIFF = {
    "font": {
        "4a": 10, "4b": 11, "4c": 12,
        "5a": 13, "5b": 14, "5c": 15,
        "6a": 16, "6a+": 17, "6b": 18, "6b+": 19,
        "6c": 20, "6c+": 21,
        "7a": 22, "7a+": 23, "7b": 24, "7b+": 25,
        "7c": 26, "7c+": 27,
        "8a": 28, "8a+": 29, "8b": 30, "8b+": 31,
        "8c": 32, "8c+": 33,
    },
    "v_grade": {
        "V0-": 10, "V0": 11, "V0+": 12,
        "V1": 13, "V1+": 14, "V2": 15,
        "V3": 16, "V3+": 17, "V4": 18, "V4+": 19,
        "V5": 20, "V5+": 21, "V6": 22, "V6+": 22.5,
        "V7": 23, "V7+": 23.5, "V8": 24, "V8+": 25,
        "V9": 26, "V9+": 26.5, "V10": 27, "V10+": 27.5,
        "V11": 28, "V11+": 28.5, "V12": 29, "V12+": 29.5,
        "V13": 30, "V13+": 30.5, "V14": 31, "V14+": 31.5,
        "V15": 32, "V15+": 32.5, "V16": 33,
    },
}

"""
DDPM model classes and generator for climb generation.

Ported from model-training/equivariant_projected_diffusion/.
Contains:
- Noiser (U-Net style denoiser)
- ClimbDDPM (diffusion wrapper)
- ClimbsFeatureScaler (data normalization)
- ClimbDDPMGenerator (generation with manifold projection)
"""
import math
import numpy as np
import pandas as pd
import sqlite3
import joblib
import os
from pathlib import Path

import torch
import torch.nn as nn
from torch import Tensor, seed
from sklearn.preprocessing import MinMaxScaler

from pathlib import Path
torch.set_printoptions(linewidth=300)


# ---------------------------------------------------------------------------
# Utility
# ---------------------------------------------------------------------------

def clear_compile_keys(filepath: Path | str, map_loc: str = "cpu") -> dict:
    """Strip torch.compile prefixes from state dict keys."""
    state_dict = torch.load(filepath, map_location=map_loc)
    new_state_dict = {}
    prefix = "_orig_mod."
    for k, v in state_dict.items():
        new_state_dict[k.removeprefix(prefix)] = v
    return new_state_dict

def zero_com(climbs: Tensor, dim: int)->Tensor:
    """Perform Zero-Center-Of-Mass transformation on a batched input climbs Tensor of shape [B,S,dim], allowing the model being trained to be translation-invariant."""
    new = climbs.clone()
    com = new[:,:,:dim].mean(dim=1, keepdim=True)
    new[:,:,:dim] -= com
    return new

# ---------------------------------------------------------------------------
# Grade lookup table
# ---------------------------------------------------------------------------

GRADE_TO_DIFF = {
    "font": {
        "4a": 10, "4b": 11, "4c": 12,
        "5a": 13, "5b": 14, "5c": 15,
        "6a": 16, "6a+": 17, "6b": 18, "6b+": 19,
        "6c": 20, "6c+": 21,
        "7a": 22, "7a+": 23, "7b": 24, "7b+": 25,
        "7c": 26, "7c+": 27,
        "8a": 28, "8a+": 29, "8b": 30, "8b+": 31,
        "8c": 32, "8c+": 33,
    },
    "v_grade": {
        "V0-": 10, "V0": 11, "V0+": 12,
        "V1": 13, "V1+": 14, "V2": 15,
        "V3": 16, "V3+": 17, "V4": 18, "V4+": 19,
        "V5": 20, "V5+": 21, "V6": 22, "V6+": 22.5,
        "V7": 23, "V7+": 23.5, "V8": 24, "V8+": 25,
        "V9": 26, "V9+": 26.5, "V10": 27, "V10+": 27.5,
        "V11": 28, "V11+": 28.5, "V12": 29, "V12+": 29.5,
        "V13": 30, "V13+": 30.5, "V14": 31, "V14+": 31.5,
        "V15": 32, "V15+": 32.5, "V16": 33,
    },
}

# ---------------------------------------------------------------------------
# ClimbDDPMGenerator
# ---------------------------------------------------------------------------

class ClimbDDPMGenerator():
    HAND_FEATURE_WEIGHTS = torch.tensor([1.0,1.0,0.8,0.8,2.0,0.1,0.1])
    FOOT_FEATURE_WEIGHTS = torch.tensor([1.0, 1.0, 0.05, 0.05, 1.0, 0.05, 0.05])
    NUM_ROLES = 5
    NUM_FEATURES = 7
    START_ROLE = 0
    FINISH_ROLE = 1
    HAND_ROLE = 2
    FOOT_ROLE = 3

    def __init__(
            self,
            scaler: 'ClimbsFeatureScaler',
            ddpm: 'ClimbDDPM',
        ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.scaler = scaler
        self.ddpm = ddpm
        self._cond_cache = {}
        self.holds_manifolds = {}
        self.holds_lookup = {}
        self.deterministic_noise_generator = torch.Generator(device=self.device)

        # Populate the manifolds immediately upon instantiation
        self.update_hold_manifolds()
    
    def update_hold_manifolds(self):
        """
        Fetches the latest holds from the database, scales features, 
        and rebuilds the manifold and lookup dictionaries.
        """
        with sqlite3.connect(DB_PATH) as conn:
            holds = pd.read_sql_query(
                "SELECT hold_index, x, y, pull_x, pull_y, useability, is_foot, tags, layout_id FROM holds", 
                conn
            )
            layout_ids = holds['layout_id'].unique()
        
        scaled_holds = self.scaler.transform_hold_features(holds, to_df=True)
        
        # Clear existing data in case layouts were removed from the DB
        self.holds_manifolds.clear()
        self.holds_lookup.clear()
        
        for layout_id in layout_ids:
            df = scaled_holds[scaled_holds['layout_id'] == layout_id]
            self.holds_manifolds[layout_id] = torch.tensor(
                df[['x', 'y', 'pull_x', 'pull_y', 'is_foot', 'pinch', 'flat']].values, 
                dtype=torch.float32
            )
            self.holds_lookup[layout_id] = df['hold_index'].values
    
    def log_hold_means(self, layout_id: str | None = None):
        """Log the hold means for each wall."""
        for k, manifold in self.holds_manifolds.items():
            if layout_id == None or layout_id == k:
                means = torch.mean(manifold, dim=0)
                print(f"Wall-id--{k}; Means-- x:{means[0].item()}, y:{means[1].item()}")

    def _build_cond_tensor(self, n, diff, angle):
        cache_key = (diff, angle)
        if cache_key not in self._cond_cache:
            row = np.array([[diff, 3.0, 1000, float(angle)]])
            scaled = self.scaler.transform_climb_features(pd.DataFrame(row, columns=['grade','quality','ascents','angle']))
            self._cond_cache[cache_key] = scaled
        base = self._cond_cache[cache_key]
        tiled = np.tile(base, (n,1))
        return torch.tensor(tiled, device=self.device, dtype=torch.float32)
    
    def _get_offset_manifold(self, layout_id: str, x_offset: float | None)-> Tensor:
        """Method for offsetting the current holds-manifold such that mean-x and mean-y is 0"""
        offset_manifold = self.holds_manifolds[layout_id].clone()
        means = torch.mean(offset_manifold, dim=0)

        if x_offset is None:
            x_offset = torch.clamp(torch.randn(size=(1,), generator=self.deterministic_noise_generator),-1.0,1.0).item()/2
        
        _range = (torch.max(offset_manifold[:,0])-torch.min(offset_manifold[0])).item()
        x_offset *= _range/2
        print(x_offset, _range/2)

        x_offset +=means[0].item()
        y_offset = means[1].item()
        
        offset_manifold[:,0] -= x_offset
        offset_manifold[:,1] -= y_offset

        return offset_manifold
    
    def _projection_strength(self, t: Tensor, t_start_projection: float = 0.8):
        """Calculate the weight to assign to the projected holds based on the timestep."""
        assert t_start_projection <= 0.8
        a = (t_start_projection-t)/t_start_projection
        if t_start_projection > 0.5:
            strength = 1 + torch.cos((a+0.5)*torch.pi)
        else:
            strength = 1 - torch.cos(a*torch.pi/2)
        return torch.where(t > t_start_projection, torch.zeros_like(t), strength).unsqueeze(2)

    def _get_nearest_manifold_neighbors(self, flat_climbs: Tensor, offset_manifold: Tensor) -> tuple[Tensor, Tensor]:
        """
        Finds the nearest manifold distances and indices for hands and feet separately.
        
        Args:
            flat_climbs: (N, H) flattened predicted holds (N can be B*S or G*B*S)
            offset_manifold: (M, self.NUM_FEATURES) manifold of available holds
        Returns:
            min_dists: (N,) tensor of nearest neighbor distances
            idx: (N,) tensor of nearest neighbor indices
        """
        is_hand_mask = (flat_climbs[:, -2] < 0.85)
        is_foot_mask = ~is_hand_mask
        
        features = flat_climbs[:, :self.NUM_FEATURES]
        
        N = flat_climbs.shape[0]
        idx = torch.empty(N, dtype=torch.long, device=flat_climbs.device)
        min_dists = torch.empty(N, dtype=torch.float32, device=flat_climbs.device)
        
        # --- Project Handholds ---
        if is_hand_mask.any():
            hand_dists = torch.cdist(
                features[is_hand_mask] * self.HAND_FEATURE_WEIGHTS.unsqueeze(0),
                offset_manifold * self.HAND_FEATURE_WEIGHTS.unsqueeze(0)
            )
            vals, indices = hand_dists.min(dim=1)
            idx[is_hand_mask] = indices
            min_dists[is_hand_mask] = vals
            
        # --- Project Footholds ---
        if is_foot_mask.any():
            foot_dists = torch.cdist(
                features[is_foot_mask] * self.FOOT_FEATURE_WEIGHTS.unsqueeze(0),
                offset_manifold * self.FOOT_FEATURE_WEIGHTS.unsqueeze(0)
            )
            vals, indices = foot_dists.min(dim=1)
            idx[is_foot_mask] = indices
            min_dists[is_foot_mask] = vals
            
        return min_dists, idx
    
    def _project_onto_manifold(self, gen_climbs: Tensor, offset_manifold: Tensor)-> Tensor:
        """
            Project each generated hold to its nearest neighbor on the hold manifold.
            
            Args:
                gen_climbs: (B, S, H) predicted clean holds
                offset_manifold: (M, self.NUM_FEATURES) manifold of available holds to snap to
            Returns:
                projected: (B, S, H) each hold snapped to nearest manifold point
        """
        B, S, H = gen_climbs.shape
        flat_climbs = gen_climbs.reshape(-1, H)
        null_mask = (flat_climbs[:, -1] < 0.95)
        
        _, idx = self._get_nearest_manifold_neighbors(flat_climbs, offset_manifold)

        projected_features = offset_manifold[idx] * null_mask.unsqueeze(1)
        
        return torch.cat([projected_features.reshape(B, S, -1), gen_climbs[:, :, self.NUM_FEATURES:]], dim=2)
    
    def _project_onto_indices(self, gen_climbs: Tensor, offset_manifold: Tensor, layout_id: str) -> list[list[list[int]]]:
        """Project climb onto the final hold indices and assign integer role values (and remove null holds and duplicate holds)"""
        B, S, H = gen_climbs.shape

        # Safely convert roles to numpy array
        roles = torch.argmax(gen_climbs[:, :, self.NUM_FEATURES:], dim=2).detach().cpu().numpy()

        flat_climbs = gen_climbs.reshape(-1, H)

        print(f"roles: {roles}")
        print(f"holds: {flat_climbs}")
        
        _, idx = self._get_nearest_manifold_neighbors(flat_climbs, offset_manifold)

        y_vals = offset_manifold[idx, 1].detach().cpu().numpy().reshape(B, S)

        # Lookup holds based on the computed indices
        # Moving idx to CPU is usually safer if holds_lookup is a numpy array or a python list
        if not isinstance(self.holds_lookup[layout_id], torch.Tensor):
            idx = idx.cpu().numpy()
            
        holds = self.holds_lookup[layout_id][idx]
        holds = holds.reshape(B, S)
        
        # Ensure holds is a numpy array before stacking with roles
        if isinstance(holds, torch.Tensor):
            holds = holds.detach().cpu().numpy()
        
        climbs = np.stack([holds, roles], axis=2)
        
        # Stack holds, roles, AND y_vals together (shape: B, S, 3)
        climbs = np.stack([holds, roles, y_vals], axis=2)
        
        deduped_climbs = []
        for c in climbs:
            valid_mask = c[:, 1] != 4
            c_valid = c[valid_mask]

            c_sorted = c_valid[c_valid[:, 1].argsort()]
            _, unique_indices = np.unique(c_sorted[:, 0], return_index=True)
            c_deduped = c_sorted[unique_indices]

            # --- Enforce Finish Hold (Highest y-value) ---
            if not np.any(c_deduped[:, 1] == self.FINISH_ROLE):
                # c_deduped[:, 2] is the y_val column
                max_y_idx = np.argmax(c_deduped[:, 2])
                c_deduped[max_y_idx, 1] = self.FINISH_ROLE
            
            # --- Enforce Start Hold (Lowest y-value handhold) ---
            if not np.any(c_deduped[:, 1] == self.START_ROLE):
                # Add "is_foot" as a float, to ensure that the lowest-value idx is of a handhold, without fucking up the index.
                min_y_idx = np.argmin(c_deduped[:, 2]+(c_deduped[:,1] == self.FOOT_ROLE).astype(np.float32))
                # Ensure we don't overwrite the finish hold if it's a 1-hold climb (edge case)
                if min_y_idx != np.argmax(c_deduped[:, 2]) or len(c_deduped) == 1:
                    c_deduped[min_y_idx, 1] = self.START_ROLE

            # Drop the y_val column so we return exactly what your downstream code expects: [hold_index, role]
            final_climb = c_deduped[:, :2].astype(int).tolist()
            deduped_climbs.append(final_climb)

        return deduped_climbs
    
    def _find_optimal_translation(
        self,
        gen_climbs: Tensor,           # (B, S, H)
        offset_manifold: Tensor,      # (M, H)
        x_offsets: Tensor,            # (Nx,)
        y_offsets: Tensor,            # (Ny,)
    ) -> tuple[Tensor, Tensor]:
        """
        Find the (dx, dy) translation per batch item that minimises total
        nearest-neighbour projection distance onto the hold manifold.
        """
        B, S, H = gen_climbs.shape
        
        null_mask = (gen_climbs[:, :, -1] < 0.95).float()
        Nx = x_offsets.shape[0]
        Ny = y_offsets.shape[0]

        # Build a (Nx, Ny, H) manifold shift table
        shifts = torch.zeros(Nx, Ny, H, device=gen_climbs.device)
        shifts[:, :, 0] = x_offsets.unsqueeze(1)   
        shifts[:, :, 1] = y_offsets.unsqueeze(0)   
        
        G = Nx * Ny
        shifts = shifts.reshape(G, H)         

        # Translate climbs: (1, B, S, H) + (G, 1, 1, H) -> (G, B, S, H)
        translated = gen_climbs.unsqueeze(0) + shifts.view(G, 1, 1, H)

        # 1. Flatten all the way down to 2D for our helper: (G*B*S, H)
        flat_translated = translated.reshape(-1, H)

        # 2. Call the upgraded helper function (we only care about the distances here)
        min_dists, _ = self._get_nearest_manifold_neighbors(flat_translated, offset_manifold)

        # 3. Reshape distances back to the 3D grid layout: (G, B, S)
        batch_dist = min_dists.reshape(G, B, S)
        
        # Apply the null mask (broadcast from B, S to G, B, S)
        batch_dist = batch_dist * null_mask.unsqueeze(0)

        # Sum distances across the sequence dimension
        total_dist = batch_dist.sum(dim=2)                      # (G, B)

        # Best grid point per batch item
        best_g = total_dist.argmin(dim=0)                       # (B,)
        best_dx = x_offsets[best_g // Ny]
        best_dy = y_offsets[best_g % Ny]

        return best_dx, best_dy

    def _project_onto_indices_with_translation(
        self,
        gen_climbs: Tensor,
        offset_manifold: Tensor,
        layout_id: str,
        x_offsets: Tensor | None = None,
        y_offsets: Tensor | None = None,
    ) -> list[list[list[int]]]:

        if x_offsets is None:
            x_offsets = torch.linspace(-0.5, 0.5, 51, device=gen_climbs.device)
        if y_offsets is None:
            y_offsets = torch.linspace(-0.5, 0.5, 51, device=gen_climbs.device)

        best_dx, best_dy = self._find_optimal_translation(
            gen_climbs, offset_manifold, x_offsets, y_offsets
        )

        # Apply per-climb optimal translation  (B, S, H)
        B, S, H = gen_climbs.shape
        translation = torch.zeros(B, 1, H, device=gen_climbs.device)
        translation[:, 0, 0] = best_dx   # x col
        translation[:, 0, 1] = best_dy   # y col  (pull_x/pull_y cols 1,3 left alone)
        translated_climbs = gen_climbs + translation

        # Now do the standard index projection on the translated climbs
        return self._project_onto_indices(translated_climbs, offset_manifold, layout_id)
    
    @torch.no_grad()
    def generate(
        self,
        layout_id: str,
        n: int,
        angle: int,
        grade: str,
        diff_scale: str,
        timesteps: int,
        deterministic: bool,
        t_start_projection: float,
        x_offset: float | None,
        guidance_value: float,
        seed: int,
    )->list[list[list[int]]]:
        """
            Generate a climb or batch of climbs with the given conditions using the standard DDPM iterative denoising process.

            :param layout_id: The layout id on which to generate the climb.
            :type layout_id: str
            :param n: The number of climbs to generate.
            :type n: int
            :param angle: The current wall angle.
            :type angle: int
            :param grade: The desired grade.
            :type grade: str
            :param diff_scale: The desired difficulty scale (V-scale or Font).
            :type diff_scale: str
            :param timesteps: Model setting: Number of diffusion timesteps to run. Higher results in better quality (Should be a divisor of 100 to retain markovian properties)
            :type timesteps: int
            :param deterministic: Whether to use the original noise vector in successive diffusion steps, or use a new noise vector each time.
            :type deterministic: bool
            :param t_start_projection: Point in the generation process to begin the projection steps. Earlier is better but more expensive.
            :type t_start_projection: float
            :param x_offset: Offset the climb on the X-axis.
            :type x_offset: float | None
            :param guidance_value: The guidance value to use for CFG generation.
            :type guidance_value: float
            :param seed: The random integer used to seed deterministic climb generation
            :type seed: int
            :return: A set of generated climbs according to the specified 
            :rtype: list[list[list[int]]]
        """
        # Seed Noise Generator
        if deterministic:
            self.deterministic_noise_generator.manual_seed(seed)
        
        # Handle manifold offset
        auto = True if x_offset is None else False
        offset_manifold = self._get_offset_manifold(layout_id, x_offset)

        # CORE LOGIC
        diff = GRADE_TO_DIFF[diff_scale][grade]
        cond_t = self._build_cond_tensor(n, diff, angle)
        x_t = torch.randn((n, 20, self.NUM_ROLES+self.NUM_FEATURES), device=self.device, generator=self.deterministic_noise_generator) if deterministic else torch.randn((n, 20, self.NUM_ROLES+self.NUM_FEATURES), device=self.device)
        noisy = x_t.clone()
        t_tensor = torch.ones((n,1), device=self.device)
        
        for _ in range(0, timesteps):
            gen_climbs = self.ddpm.predict_cfg(noisy, cond_t, t_tensor, guidance_value)

            if t_tensor[0].item() < t_start_projection:
                alpha_p = self._projection_strength(t_tensor, t_start_projection)
                projected_climbs = self._project_onto_manifold(gen_climbs, offset_manifold)
                gen_climbs = alpha_p*(projected_climbs) + (1-alpha_p)*(gen_climbs)
            
            t_tensor -= 1.0/timesteps
            noisy = self.ddpm.forward_diffusion(gen_climbs, t_tensor, noisy if deterministic else torch.randn_like(noisy))
                
        if auto:
            return self._project_onto_indices_with_translation(gen_climbs, offset_manifold, layout_id)
        return self._project_onto_indices(gen_climbs, offset_manifold, layout_id)


# ---------------------------------------------------------------------------
# Global ClimbGenerator Instance For Dependency Injection
# ---------------------------------------------------------------------------
def reset_generator():
    scaler = ClimbsFeatureScaler(
        weights_path=SCALER_WEIGHTS_PATH
    )
    ddpm = ClimbDDPM(
        model=Noiser(),
        weights_path=DDPM_WEIGHTS_PATH,
    )

    ddpm.eval()

    generator = ClimbDDPMGenerator(
        scaler=scaler,
        ddpm=ddpm
    )

    generator.log_hold_means()
    return generator

generator = reset_generator()
generator.generate(
    "layout-a611b479b219",
    1,
    40,
    "V1",
    "v_grade",
    50,
    False,
    0.04,
    None,
    3.0,
    37
)

c:\Users\EvanM\Documents\Projects\GitHub\ml-homewall-climb-generator\model-training\diffusion_utils.py:65: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.l

Wall-id--layout-0aa86d03949f; Means-- x:-0.0006921293097548187, y:-0.08444114774465561
Wall-id--layout-95a3f6e2ba1a; Means-- x:-0.43478256464004517, y:-0.11888893693685532
Wall-id--layout-f712a64fc0dc; Means-- x:-0.30434784293174744, y:-0.15998880565166473
Wall-id--layout-f6702371e300; Means-- x:-0.30434781312942505, y:-0.19692584872245789
Wall-id--layout-f311591b6b8b; Means-- x:-0.3031255602836609, y:-0.15307742357254028
Wall-id--layout-47aa8b2f3cbc; Means-- x:-0.30434781312942505, y:-0.1721336543560028
Wall-id--layout-5311e4b5fa08; Means-- x:-0.5652173757553101, y:-0.13500653207302094
Wall-id--layout-a611b479b219; Means-- x:-0.44279012084007263, y:-0.3262856900691986
0.20180253410161164 0.41299527883529663
roles: [[0 0 3 3 3 3 3 2 2 3 2 2 2 2 2 1 1 4 4 4]]
holds: tensor([[-2.8892e-01, -3.3004e-01, -2.4632e-01, -5.8607e-01,  4.6978e-03,  1.0174e-04, -7.7591e-03,  1.0101e+00,  7.9189e-03,  6.2234e-04,  5.9770e-03, -1.5725e-02],
        [-5.8917e-02, -3.4840e-01,  5.4754e-01, -4.7986e-0

[[[6, 2],
  [8, 1],
  [9, 2],
  [21, 1],
  [37, 2],
  [70, 2],
  [81, 2],
  [82, 2],
  [95, 3],
  [123, 3],
  [127, 0],
  [133, 0],
  [135, 3],
  [142, 3],
  [147, 3],
  [149, 3]]]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

def animate_climb_generation(guesses: list, filename="climb_diffusion_no_projection.gif", title="Diffusion Process, V4 @45*, No Projection", fps=15):
    """
    Creates a GIF animating the DDPM denoising process.
    
    Args:
        guesses (list): A list of numpy arrays. Each array should be shape (20, 4) 
                        representing [x, y, pull_x, pull_y] at a specific timestep.
        filename (str): Output filename for the GIF.
        fps (int): Frames per second.
    """
    n_holds = len(guesses[0])
    # Setup the figure and axis
    fig, ax = plt.subplots(figsize=(6, 8))
    
    # We set limits slightly wider than the standard -1,1 to see the noise coalescing
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_xlabel("X Position (Normalized)")
    ax.set_ylabel("Y Position (Normalized)")
    
    # Initialize graphic elements
    # Scatter plot for holds
    # We initialize with dummy data; 'c' and 's' will be updated in the loop
    dummy = np.zeros(n_holds)
    scat = ax.scatter(dummy, dummy, c=dummy, s=dummy, cmap='viridis', vmin=0, vmax=1, zorder=2)
    
    # Quiver plot for pull vectors
    # angles='xy', scale_units='xy', scale=0.5 matches your plot_climb logic
    quiv = ax.quiver(dummy, dummy, dummy, dummy, 
                     color='green', alpha=0.6,
                     angles='xy', scale_units='xy', scale=0.5,
                     width=0.005, headwidth=3, zorder=1)

    def init():
        """Initialize empty data for the animation."""
        scat.set_offsets(np.empty((0, 2)))
        quiv.set_offsets(np.empty((0, 2)))
        quiv.set_UVC(dummy, dummy)
        return scat, quiv

    def update(frame_idx):
        """Update function for each frame of the animation."""
        # Get data for the current timestep
        # Data shape: (20, 4) -> [x, y, pull_x, pull_y]
        data = guesses[frame_idx]
        
        x = data[:, 0]
        y = data[:, 1]
        pull_x = data[:, 2]
        pull_y = data[:, 3]

        # Determine which holds act as "hands" (significant pull) vs "feet"/smears
        # This matches your plot_climb logic
        is_hand = ((np.absolute(pull_y) > 0.2) | (np.absolute(pull_x) > 0.2))
        
        # Calculate sizes based on type
        sizes = np.full(len(is_hand), 20) + np.full(len(is_hand), 30) * is_hand.astype(float)
        
        # Update Scatter (Positions, Sizes, Colors)
        # Note: set_offsets expects an (N, 2) array
        scat.set_offsets(np.c_[x, y])
        scat.set_sizes(sizes)
        # We use the boolean is_hand array for color mapping (0=Purple, 1=Yellow by default)
        scat.set_array(is_hand.astype(float))
        
        # Update Quiver (Positions and Vectors)
        quiv.set_offsets(np.c_[x, y])
        # Apply the 0.2 scaling factor from your reference code
        quiv.set_UVC(pull_x * 0.2, pull_y * 0.2)
        
        # Update Title
        # Reverse the count so it looks like a countdown (Noise -> Clean) or standard count
        ax.set_title(f"{title}\n t={len(guesses)-frame_idx}/{len(guesses)}")
        
        return scat, quiv

    # Create the animation
    print(f"Generating animation with {len(guesses)} frames...")
    anim = FuncAnimation(fig, update, frames=len(guesses), init_func=init, blit=False)
    
    # Save to GIF
    writer = PillowWriter(fps=fps)
    anim.save(filename, writer=writer)
    print(f"Animation saved to {filename}")

animate_climb_generation(guesses, filename="strong_projection.gif", title="Diffusion Process, V4 @45*, Early/Strong Projection")